In [ ]:
import numpy as np
import os
import sys

sys.path.append('../../..')
path=os.path.abspath(os.path.join(os.getcwd(), "../../../outputs/kuramoto/nnc"))

import torch
import torch.optim.swa_utils as swa_utils
from torch import nn
import copy
from types import SimpleNamespace

from torch import profiler
from tqdm import tqdm

import torchsde
from core.utils import get_gpu_memory,objective,update_ema_model, adj_lattice, criticalK, order_param,  load_clean_state_dict, sample_kuramoto,loss_compute_kur,save_model
from core.policies_kur import  HyperKuramoto,  find_params
from configs.kuramoto_cfg import k_cfg
import gc
from IPython.display import clear_output

is_cuda = torch.cuda.is_available()
device = 'cuda' if is_cuda else 'cpu'

if not is_cuda:
    print("Warning: CUDA not available; falling back to CPU but this is likely to be very slow.")

In [ ]:

# Graph Setup
adj_matrix = adj_lattice(k_cfg.state_size, k_cfg.graph, k_cfg.connectivity)
kcrit, L_np = criticalK(adj_matrix, k_cfg.state_size, k_cfg.scale)
K_value = 0.01 * kcrit
# Save the adjacency matrix for testing
%store adj_matrix

# Elegant Factory Pattern for Tensors
ctx = {'device': device, 'dtype': torch.float32}
A_torch = torch.as_tensor(adj_matrix, **ctx)
L_torch = torch.as_tensor(L_np, **ctx)
dist = torch.distributions.Normal(0, 1)

print(f"Coupling Constant K: {K_value:.4f}")

In [ ]:
def train_kuramoto(layers, config, A, L):
    # Model & Optimization Setup
    model = HyperKuramoto(
        config.batch_size, config.state_size, K_value, 
        A, layers, config.skip, config.strategy, device, config.activation
    ).to(device)
    
    ema_model = copy.deepcopy(model)
    optimizer = torch.optim.AdamW(find_params(model), lr=config.lr, weight_decay=0.0)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=config.gamma)
    
    # Trackers
    stats = {k: [] for k in ['loss', 'ema_loss', 'ste', 'order_param']}
    ema_loss = 0.0
    best_score = 1e6
    ts_exp = torch.linspace(0, config.t_total, config.t_size).to(device)

    pbar = tqdm(range(config.iterations), desc=f"Kuramoto L{layers}")
    
    for i in pbar:
        # 1. Sample Dynamics
        freqs, ys_init = sample_kuramoto(
            config.state_size, config.batch_size, dist, 
            config.scale, config.scale_kin, device
        )
        model.freqs = freqs

        # 2. Forward Simulation (Neural SDE)
        ys_tray = torchsde.sdeint_adjoint(
            model, ys_init, ts_exp, method='reversible_heun', 
            dt=ts_exp[1]-ts_exp[0], adjoint_method='adjoint_reversible_heun'
        )

        # 3. Physics-Informed Metrics
        loss, ste_loss, logtrick = loss_compute_kur(
            ys_tray, config.state_size, config.batch_size, config.weight, A
        )
        ord_p = order_param(ys_tray[-1, :, 0:config.state_size], L)

        # 4. Backpropagation
        optimizer.zero_grad(set_to_none=True)
        logtrick.backward()
        optimizer.step()

        # 5. EMA & Weights Management
        model.project(config.strategy,config.clip)
        
        scheduler.step()
        update_ema_model(model, ema_model, config.alpha)
        
        # Smoothed Loss Tracking
        curr_loss = loss.item()
        ema_loss = curr_loss if i == 0 else (ema_loss * config.alpha + curr_loss * (1 - config.alpha))

        # 6. Checkpointing and Logging
        best_score = save_model(best_score, ema_loss, ema_model, model, path, config.strategy, layers)
        
        # Update progress bar 
   
        pbar.set_postfix({
            "Loss": f"{curr_loss:.4e}",
            "EMA": f"{ema_loss:.4e}",
            "OrderP": f"{ord_p.item():.4f}"
        })

        # Append data to storage
        stats['loss'].append(curr_loss)
        stats['ema_loss'].append(ema_loss)
        stats['ste'].append(ste_loss.item())
        stats['order_param'].append(ord_p.item())

        # Memory Cleanup (Once every 500 steps is enough)
        if i % 500 == 0:
            torch.cuda.empty_cache()
            gc.collect()

    return stats

In [ ]:
all_run_stats = []

for layer_count in range(1, 4):
    print(f"\n--- Training Kuramoto Strategy: {k_cfg.strategy} | Layers: {layer_count} ---")
    run_results = train_kuramoto(layer_count, k_cfg, A_torch, L_torch)
    all_run_stats.append(run_results)

print("Experiment Complete.")